# Datapreprocessing

Remove high ranking frequent words from manifold analysis to prevent contaminatation of dominant compoents

In [1]:
import os
import re
import json
import bisect
from collections import Counter
from dataclasses import dataclass, field, asdict
from functools import lru_cache, cached_property
from typing import Dict, Optional, Tuple

import torch
import numpy as np
from safetensors.torch import load_file
from transformers import AutoTokenizer

from dotenv import load_dotenv
import os
load_dotenv()
token = os.environ.get('HF_TOKEN')

try:
    import plotly.graph_objects as go
    _PLOTLY = True
except ImportError:
    _PLOTLY = False


# ── (1) Collation ────────────────────────────────────────────────────────────

def _shard_no(filename: str) -> int:
    """Extract the integer shard ordinal so sorting is numeric, not lexical."""
    return int(re.search(r"shard_(\d+)", filename).group(1))


def collate_layer(
    output_base: str,
    layer: int,
    *,
    dtype: torch.dtype | None = None,
    device: str = "cpu",
) -> torch.Tensor:
    """
    Vertically concatenate all shards of `layer` into one (Σtokens, D) tensor.

    `output_base` identifies the dataset run (e.g. OUTPUT_BASE from collection).
    Shards are ordered by their numeric ordinal so the result aligns exactly
    with the cumulative ordering encoded in metadata.jsonl.
    """
    layer_dir = os.path.join(output_base, f"layer_{layer}")
    files = sorted(
        (f for f in os.listdir(layer_dir) if f.endswith(".safetensors")),
        key=_shard_no,
    )
    if not files:
        raise FileNotFoundError(f"No shards found in {layer_dir}")

    chunks = []
    for fname in files:
        t = load_file(os.path.join(layer_dir, fname))["activations"]
        if dtype is not None:
            t = t.to(dtype)
        chunks.append(t)

    return torch.cat(chunks, dim=0).to(device)

In [88]:
# ── (2) Frequency filter result ───────────────────────────────────────────────

@dataclass
class FrequencyFilterResult:
    """
    Encodes the bidirectional correspondence between the original collated matrix
    X ∈ ℝ^{N×D} and the frequency-filtered submatrix X_parsed ∈ ℝ^{N_parsed×D}.

    Attributes
    ----------
    X_parsed : torch.Tensor, shape (N_parsed, D)
        Rows of X whose token fell below ``max_count``, in original order.
    parsed_to_original : torch.Tensor, shape (N_parsed,), dtype long
        ``parsed_to_original[i]`` is the global row index of ``X_parsed[i]``
        within the full collated matrix X.  Use this to project SVD coordinates
        back to corpus positions, or to call ``ActivationIndex.locate()``.
    original_to_parsed : torch.Tensor, shape (N,), dtype long
        Inverse map: ``original_to_parsed[j]`` is the row of ``X_parsed``
        that corresponds to row ``j`` of X.  Entries equal to ``-1`` indicate
        rows that were dropped (token frequency exceeded ``max_count``).
    n_kept : int
        Number of rows retained in X_parsed.
    n_dropped : int
        Number of rows excised due to frequency ceiling.
    max_count : int
        The ceiling applied during filtering.
    """
    X_parsed: torch.Tensor
    parsed_to_original: torch.Tensor
    original_to_parsed: torch.Tensor
    n_kept: int
    n_dropped: int
    max_count: int


# ── (3) Reverse index: global row → token + textual context ──────────────────

@dataclass
class TokenHit:
    global_index:   int
    shard:          int
    sample:         int
    row:            int
    token_position: int
    token_id:       int
    token_str:      str
    sample_length:  int
    snippet:        str
    full_text:      str

    # ── Token activation block ────────────────────────────────────────────────────

@dataclass
class TokenActivationBlock:
    """
    All activation rows corresponding to a single token type.

    Attributes
    ----------
    activations : torch.Tensor, shape (K, D)
        The K rows of the collated matrix whose token equals ``token_id``,
        in corpus order.
    global_indices : torch.Tensor, shape (K,), dtype long
        Global row indices into the full collated matrix X; pass any element
        directly to ``ActivationIndex.locate()`` to recover textual context.
    token_id : int
        Vocabulary integer ID of the token.
    token_str : str
        Human-readable surface form as rendered by the tokenizer.
    frequency : int
        Corpus-wide occurrence count (equals K when X is the full matrix;
        may be smaller if X is a filtered submatrix).
    rank : int
        Frequency rank used to select this token (1 = most common).
    """
    activations:    torch.Tensor
    global_indices: torch.Tensor
    token_id:       int
    token_str:      str
    frequency:      int
    rank:           int
    

@dataclass
class IndexActivationBlock:
    """
    All activation rows corresponding to a fixed token position across every
    corpus sample of sufficient length.

    Attributes
    ----------
    activations : torch.Tensor, shape (K, D)
        Rows of X drawn from position ``token_position`` within each
        contributing sample, in corpus order.
    global_indices : torch.Tensor, shape (K,), dtype long
        Global row indices into the full collated matrix X; pass any element
        to ``ActivationIndex.locate()`` to recover textual context.
    token_ids : torch.Tensor, shape (K,), dtype long
        Vocabulary ID of the token occupying ``token_position`` in each
        contributing sample.  Heterogeneous across samples — unlike
        ``TokenActivationBlock``, no single token type is implied.
    token_strs : list[str]
        Human-readable surface forms corresponding element-wise to
        ``token_ids``.
    token_position : int
        The 0-based intra-sample index from which all rows were drawn.
    n_samples : int
        Number of corpus samples whose length exceeded ``token_position``
        before any frequency filtering; equals ``activations.shape[0]``
        when ``original_to_parsed`` is None.
    """
    activations:    torch.Tensor
    global_indices: torch.Tensor
    token_ids:      torch.Tensor
    token_strs:     list[str]
    token_position: int
    n_samples:      int
    

class ActivationIndex:
    """
    Resolves any row of the collated activation tensor to its originating
    token and surrounding text, without holding activations in memory.

    Extended capabilities
    ---------------------
    * ``build_token_frequencies()``  — corpus-wide token-ID counts (memoised).
    * ``all_token_ids()``            — (total_tokens,) tensor of per-row token IDs.
    * ``filter_by_frequency()``      — frequency-ceiling filter with bidirectional
                                       index mapping between X and X_parsed.
    * ``sif_weight()``               — Smooth Inverse Frequency row-scaling
                                       (Arora et al., 2017).
    * ``plot_token_frequencies()``   — interactive Plotly rank–frequency chart
                                       with per-token hover annotations.
    """

    def __init__(
        self,
        output_base: str,
        model_name: str = "google/gemma-2-2b",
        max_seq_len: int = 4096,
    ):
        self.base        = output_base
        self.max_seq_len = max_seq_len
        self.tokenizer   = AutoTokenizer.from_pretrained(model_name)
        self._load_metadata()

    # ── Metadata ──────────────────────────────────────────────────────────────

    def _load_metadata(self) -> None:
        self.samples: list[dict] = []
        self._ends: list[int]   = []
        running = 0
        path = os.path.join(self.base, "metadata.jsonl")
        with open(path, encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                rec["global_start"] = running
                running += rec["length"]
                self.samples.append(rec)
                self._ends.append(running)
        self.total_tokens = running

    @lru_cache(maxsize=32)
    def _text_lines(self, shard: int) -> tuple[str, ...]:
        path = os.path.join(self.base, "text-samples", f"shard_{shard}.jsonl")
        with open(path, encoding="utf-8") as f:
            return tuple(json.loads(l)["text"] for l in f)

    def _sample_text(self, rec: dict) -> str:
        return self._text_lines(rec["shard"])[rec["row"]]

    @lru_cache(maxsize=4096)
    def _sample_tokens(self, sample_id: int) -> tuple[int, ...]:
        rec  = self.samples[self._sample_pos[sample_id]]
        text = self._sample_text(rec)
        ids  = self.tokenizer(
            text, truncation=True, max_length=self.max_seq_len
        )["input_ids"]
        return tuple(ids)

    @property
    def _sample_pos(self) -> dict[int, int]:
        if not hasattr(self, "_sp"):
            self._sp = {r["sample"]: i for i, r in enumerate(self.samples)}
        return self._sp

    # ── Token-ID sequence ─────────────────────────────────────────────────────

    @cached_property
    def _token_id_tensor(self) -> torch.Tensor:
        """
        (total_tokens,) integer tensor: ``_token_id_tensor[i]`` is the token_id
        of global row i in the collated activation matrix.

        Constructed by iterating samples in metadata order and concatenating
        their tokenised ID sequences.  Memoised as a ``cached_property``; the
        first call is O(total_tokens) in time and memory.
        """
        parts: list[list[int]] = []
        for rec in self.samples:
            parts.append(list(self._sample_tokens(rec["sample"])))
        return torch.tensor(
            [tid for seq in parts for tid in seq], dtype=torch.long
        )

    def all_token_ids(self) -> torch.Tensor:
        """
        Public accessor for ``_token_id_tensor``.
        Returns a (total_tokens,) long tensor; safe to index directly into
        a frequency lookup table.
        """
        return self._token_id_tensor

    # ── Frequency map ─────────────────────────────────────────────────────────

    def build_token_frequencies(self) -> Counter:
        """
        Corpus-wide count of each token_id.

        Iterates all samples exactly once, exploiting the ``_sample_tokens``
        LRU cache so the tokenizer is not invoked redundantly.  The result is
        memoised on the instance; subsequent calls are O(1).

        Returns
        -------
        Counter
            Mapping token_id (int) → occurrence count (int).
        """
        if hasattr(self, "_freq"):
            return self._freq
        c: Counter = Counter()
        for rec in self.samples:
            c.update(self._sample_tokens(rec["sample"]))
        self._freq = c
        return self._freq

    # ── Frequency-ceiling filter with bidirectional index mapping ─────────────

    def filter_by_frequency(
        self,
        X: torch.Tensor,
        max_count: int = 16_000,
    ) -> FrequencyFilterResult:
        """
        Construct X_parsed by excising rows whose token_id appears more than
        ``max_count`` times in the corpus, and return the complete bidirectional
        index correspondence between X and X_parsed.

        Parameters
        ----------
        X : torch.Tensor, shape (N, D)
            The collated activation matrix; N must equal ``self.total_tokens``.
        max_count : int
            Frequency ceiling (inclusive).  A token appearing exactly
            ``max_count`` times is retained; one appearing ``max_count + 1``
            times is dropped.

        Returns
        -------
        FrequencyFilterResult
            See dataclass docstring for the full field specification.

        Notes
        -----
        The count lookup is vectorised: a dense integer tensor indexed by
        token_id is constructed from the Counter, avoiding a Python-level loop
        over N rows.  This is O(N) in time after ``build_token_frequencies()``.
        """
        if X.shape[0] != self.total_tokens:
            raise ValueError(
                f"X has {X.shape[0]} rows but index covers {self.total_tokens} "
                "tokens.  Ensure X was produced by collate_layer over the same "
                "output_base."
            )
        freq       = self.build_token_frequencies()
        token_ids  = self.all_token_ids()          # (N,)

        # Build a dense lookup tensor: count_lut[token_id] = frequency.
        # Out-of-vocabulary token IDs (absent from the corpus) receive count 0.
        max_tid    = int(token_ids.max().item()) + 1
        count_lut  = torch.zeros(max_tid, dtype=torch.long)
        for tid, cnt in freq.items():
            if tid < max_tid:
                count_lut[tid] = cnt
        row_counts = count_lut[token_ids]          # (N,) vectorised lookup

        mask = row_counts <= max_count             # True ⇒ retain

        # parsed_to_original[i] = global row index in X of X_parsed[i]
        parsed_to_original = torch.nonzero(mask, as_tuple=False).flatten()

        # original_to_parsed[j] = row in X_parsed that maps back to X[j];
        # -1 signals a dropped row, consistent with scatter-index conventions.
        original_to_parsed = torch.full((X.shape[0],), -1, dtype=torch.long)
        original_to_parsed[parsed_to_original] = torch.arange(
            parsed_to_original.shape[0], dtype=torch.long
        )

        n_kept    = int(parsed_to_original.shape[0])
        n_dropped = X.shape[0] - n_kept

        return FrequencyFilterResult(
            X_parsed           = X[parsed_to_original].contiguous(),
            parsed_to_original = parsed_to_original,
            original_to_parsed = original_to_parsed,
            n_kept             = n_kept,
            n_dropped          = n_dropped,
            max_count          = max_count,
        )

    # ── SIF weighting ─────────────────────────────────────────────────────────

    def sif_weight(
        self,
        X: torch.Tensor,
        token_ids: torch.Tensor,
        a: float = 1e-3,
    ) -> torch.Tensor:
        """
        Apply Smooth Inverse Frequency weighting (Arora et al., 2017) to the
        activation matrix.

        For each row i with token T_i, the per-token scalar weight is:

            w(T_i) = a / (a + p(T_i)),   a ≈ 1e-3

        where p(T_i) = count(T_i) / total_tokens is the unigram probability.
        The returned matrix X̃ has the same shape as X, with each row scaled by
        its corresponding weight:

            X̃_i = w(T_i) · X_i

        High-frequency tokens (p(T) → 1) receive weights approaching zero;
        hapax legomena receive weights approaching one.  This is the activation-
        space analogue of IDF and approximates PPMI's cancellation of marginal
        probabilities.

        Parameters
        ----------
        X : torch.Tensor, shape (N, D)
            Activation matrix to be weighted.  Typically the output of
            ``collate_layer``, or ``FrequencyFilterResult.X_parsed``.
        token_ids : torch.Tensor, shape (N,), dtype long
            Token ID for each row of X.  Pass ``self.all_token_ids()`` for
            the full collated matrix, or index it with
            ``FrequencyFilterResult.parsed_to_original`` for X_parsed:
            ``self.all_token_ids()[result.parsed_to_original]``.
        a : float
            SIF smoothing parameter; Arora et al. recommend a ∈ [1e-4, 1e-3].

        Returns
        -------
        torch.Tensor, shape (N, D)
            SIF-weighted activation matrix X̃, on the same device and dtype as X.
        """
        freq  = self.build_token_frequencies()
        total = float(self.total_tokens)

        # Vectorised probability lookup — same dense-tensor strategy as filter.
        max_tid   = int(token_ids.max().item()) + 1
        prob_lut  = torch.zeros(max_tid, dtype=X.dtype, device=X.device)
        for tid, cnt in freq.items():
            if tid < max_tid:
                prob_lut[tid] = cnt / total

        p = prob_lut[token_ids]                    # (N,)
        w = a / (a + p)                            # (N,)  ∈ (0, 1]
        return X * w.unsqueeze(1)

    # ── Plotly frequency visualisation ────────────────────────────────────────

    def plot_token_frequencies(
        self,
        top_k: int = 5_000,
        max_count_threshold: Optional[int] = None,
        log_scale: bool = True,
    ) -> "go.Figure":
        """
        Render an interactive rank–frequency chart of the ``top_k`` most common
        tokens.  Hovering over each bar reveals the token string, its integer ID,
        and its corpus count.

        Parameters
        ----------
        top_k : int
            Number of highest-frequency tokens to display.
        max_count_threshold : int or None
            When provided, draws a horizontal reference line at this count,
            visually delineating which tokens would be excised by
            ``filter_by_frequency(max_count=max_count_threshold)``.
        log_scale : bool
            Whether to render the y-axis on a logarithmic scale (recommended;
            token frequency distributions are approximately Zipfian).

        Returns
        -------
        plotly.graph_objects.Figure
        """
        if not _PLOTLY:
            raise ImportError(
                "plotly is required for visualisation: pip install plotly"
            )

        freq  = self.build_token_frequencies()
        items = freq.most_common(top_k)            # [(token_id, count), ...]

        token_ids_sorted = [tid for tid, _   in items]
        counts           = [cnt for _,   cnt in items]
        token_strs       = [
            repr(self.tokenizer.convert_ids_to_tokens(tid) or f"<id:{tid}>")
            for tid in token_ids_sorted
        ]

        bar = go.Bar(
            x            = list(range(len(counts))),
            y            = counts,
            marker_color = "steelblue",
            customdata   = list(zip(token_ids_sorted, token_strs, counts)),
            hovertemplate = (
                "<b>Token:</b> %{customdata[1]}<br>"
                "<b>Token ID:</b> %{customdata[0]}<br>"
                "<b>Count:</b> %{customdata[2]:,}<br>"
                "<b>Rank:</b> %{x}<extra></extra>"
            ),
        )

        fig = go.Figure(data=[bar])

        if max_count_threshold is not None:
            fig.add_hline(
                y            = max_count_threshold,
                line_dash    = "dash",
                line_color   = "crimson",
                annotation_text  = f"Frequency ceiling ({max_count_threshold:,})",
                annotation_position = "top right",
            )

        fig.update_layout(
            title      = f"Token Frequency Distribution — top {top_k:,} types",
            xaxis_title = "Rank (descending frequency)",
            yaxis_title = "Corpus occurrence count",
            bargap     = 0.0,
            hovermode  = "x unified",
        )

        if log_scale:
            fig.update_yaxes(type="log", title_text="Corpus occurrence count (log)")

        return fig

    # ── Point lookup ──────────────────────────────────────────────────────────

    def locate(self, global_index: int, *, snippet_window: int = 16) -> TokenHit:
        """Resolve a single row of the collated tensor to its token and context."""
        if not 0 <= global_index < self.total_tokens:
            raise IndexError(
                f"{global_index} out of range [0, {self.total_tokens})"
            )
        si        = bisect.bisect_right(self._ends, global_index)
        rec       = self.samples[si]
        token_pos = global_index - rec["global_start"]
        ids       = self._sample_tokens(rec["sample"])

        if len(ids) != rec["length"]:
            raise RuntimeError(
                f"Length mismatch for sample {rec['sample']}: "
                f"stored {rec['length']}, re-tokenised {len(ids)}. "
                "Verify model_name and max_seq_len match collection."
            )

        token_id = ids[token_pos]
        lo = max(0, token_pos - snippet_window)
        hi = min(len(ids), token_pos + snippet_window + 1)

        return TokenHit(
            global_index   = global_index,
            shard          = rec["shard"],
            sample         = rec["sample"],
            row            = rec["row"],
            token_position = token_pos,
            token_id       = token_id,
            token_str      = self.tokenizer.convert_ids_to_tokens(token_id),
            sample_length  = rec["length"],
            snippet        = self.tokenizer.decode(ids[lo:hi]),
            full_text      = self._sample_text(rec),
        )

    def get_activation(self, global_index: int, layer: int) -> torch.Tensor:
        """
        Fetch the (D,) activation vector for a single row without collating
        the whole layer.
        """
        si        = bisect.bisect_right(self._ends, global_index)
        rec       = self.samples[si]
        token_pos = global_index - rec["global_start"]
        shard_row = rec["offset"] + token_pos
        path = os.path.join(
            self.base, f"layer_{layer}", f"shard_{rec['shard']}.safetensors"
        )
        return load_file(path)["activations"][shard_row]
    def gather_token_activations(
        self,
        X: torch.Tensor,
        *,
        rank: int = 1,
        token_id: Optional[int] = None,
        row_map: Optional[torch.Tensor] = None,
    ) -> TokenActivationBlock:
        """
        Collect every row of X whose token matches a given vocabulary type,
        identified either by frequency rank or by explicit token_id.
    
        Parameters
        ----------
        X : torch.Tensor, shape (N, D)
            Activation matrix — either the full collated matrix or
            ``FrequencyFilterResult.X_parsed``.
        rank : int
            1-based frequency rank used to select the token when ``token_id``
            is not supplied.  rank=1 selects the single most frequent token.
            Ignored when ``token_id`` is provided explicitly.
        token_id : int or None
            If supplied, selects this vocabulary ID directly, bypassing rank
            resolution.
        row_map : torch.Tensor, shape (N,), dtype long — or None
            When X is a filtered submatrix (``FrequencyFilterResult.X_parsed``),
            pass ``FrequencyFilterResult.parsed_to_original`` so that the returned
            ``global_indices`` reference the original collated matrix rather than
            the submatrix.  When X is the full matrix, leave as None.
    
        Returns
        -------
        TokenActivationBlock
            See dataclass docstring.
    
        Raises
        ------
        ValueError
            If ``rank`` exceeds the number of distinct token types observed.
        """
        freq = self.build_token_frequencies()
    
        if token_id is None:
            if rank < 1 or rank > len(freq):
                raise ValueError(
                    f"rank={rank} is outside [1, {len(freq)}] "
                    f"(only {len(freq)} distinct token types observed)."
                )
            # most_common returns a list; index is 0-based
            token_id, _ = freq.most_common(rank)[rank - 1]
    
        frequency  = freq[token_id]
        token_str  = self.tokenizer.convert_ids_to_tokens(token_id) or f"<id:{token_id}>"
    
        # Resolve which row indices (within X) carry this token_id.
        # When X is filtered, _token_id_tensor must be sliced by row_map first
        # so that its length aligns with X.shape[0].
        if row_map is not None:
            local_token_ids = self._token_id_tensor[row_map]   # (N_parsed,)
        else:
            local_token_ids = self._token_id_tensor             # (N,)
    
        local_mask     = local_token_ids == token_id            # (N,) bool
        local_indices  = torch.nonzero(local_mask, as_tuple=False).flatten()
    
        # global_indices map back to the original collated matrix in all cases.
        global_indices = row_map[local_indices] if row_map is not None else local_indices
    
        return TokenActivationBlock(
            activations    = X[local_indices].contiguous(),
            global_indices = global_indices,
            token_id       = int(token_id),
            token_str      = str(token_str),
            frequency      = frequency,
            rank           = rank,
        )
        def gather_index_activations(
            self,
            X: torch.Tensor,
            token_position: int,
            *,
            original_to_parsed: Optional[torch.Tensor] = None,
        ) -> IndexActivationBlock:
            """
            Collect every row of X whose token occupies a given positional index
            within its originating sample.
        
            Parameters
            ----------
            X : torch.Tensor, shape (N, D)
                Activation matrix — either the full collated matrix or
                ``FrequencyFilterResult.X_parsed``.
            token_position : int
                0-based token index within each sample.  Only samples whose stored
                length exceeds this index contribute rows.
            original_to_parsed : torch.Tensor, shape (N_full,), dtype long — or None
                When X is a filtered submatrix (``FrequencyFilterResult.X_parsed``),
                pass ``FrequencyFilterResult.original_to_parsed`` so that global
                indices are translated to local row positions within X.  Entries
                equal to -1 (rows excised by frequency filtering) are excluded
                automatically, causing ``activations.shape[0]`` to fall below
                ``n_samples``.  When X is the full collated matrix, leave as None.
        
            Returns
            -------
            IndexActivationBlock
                See dataclass docstring.
        
            Raises
            ------
            ValueError
                If ``token_position`` is negative, or if no corpus sample has
                length exceeding ``token_position``.
            """
            if token_position < 0:
                raise ValueError(
                    f"token_position must be non-negative; received {token_position}."
                )
        
            # Accumulate the global row index for every sample that reaches this
            # position.  global_start + token_position maps directly into the
            # collated tensor because rows are laid out in sample-then-position order.
            global_idx_list: list[int] = [
                rec["global_start"] + token_position
                for rec in self.samples
                if rec["length"] > token_position
            ]
        
            n_samples = len(global_idx_list)
            if n_samples == 0:
                raise ValueError(
                    f"No corpus sample has length > {token_position}; "
                    "token_position is out of range for the entire corpus."
                )
        
            global_indices = torch.tensor(global_idx_list, dtype=torch.long)
        
            # Token IDs at this position are heterogeneous across samples;
            # retrieve them via the cached token-ID tensor rather than re-tokenising.
            token_ids  = self._token_id_tensor[global_indices]          # (K,)
            token_strs = [
                self.tokenizer.convert_ids_to_tokens(int(tid)) or f"<id:{int(tid)}>"
                for tid in token_ids
            ]
        
            # Resolve local row indices within X.
            # For the full matrix, local position equals global position.
            # For a filtered submatrix, original_to_parsed provides the mapping,
            # and -1 entries flag rows that were excised during filtering.
            if original_to_parsed is not None:
                local_indices = original_to_parsed[global_indices]       # (K,)
                keep          = local_indices >= 0                        # bool mask
                local_indices  = local_indices[keep]
                global_indices = global_indices[keep]
                token_ids      = token_ids[keep]
                token_strs     = [s for s, k in zip(token_strs, keep.tolist()) if k]
            else:
                local_indices = global_indices
        
            return IndexActivationBlock(
                activations    = X[local_indices].contiguous(),
                global_indices = global_indices,
                token_ids      = token_ids,
                token_strs     = token_strs,
                token_position = token_position,
                n_samples      = n_samples,
            )            

In [ ]:
    
BASE, LAYER = "/vast/temp/enguye17/fw-acts/data/", 12

# Collate
X = collate_layer(BASE, LAYER, dtype=torch.float32)
print("collated:", tuple(X.shape), X.dtype)

idx = ActivationIndex(BASE)

# ── Frequency map ──────────────────────────────────────────────────────
freq = idx.build_token_frequencies()
print(f"Vocabulary covered: {len(freq):,} distinct token IDs")
print(f"10 most common: {freq.most_common(10)}")

# ── Filter (X → X_parsed) ──────────────────────────────────────────────
result = idx.filter_by_frequency(X, max_count=16_000)
print(
    f"Filter: kept {result.n_kept:,} / dropped {result.n_dropped:,} rows "
    f"(ceiling = {result.max_count:,})"
)

# Bidirectional round-trip sanity check on a retained row
sample_parsed_idx = 0
original_idx      = int(result.parsed_to_original[sample_parsed_idx].item())
roundtrip         = int(result.original_to_parsed[original_idx].item())
assert roundtrip == sample_parsed_idx, "Round-trip index mismatch"

# Resolve the first retained token via the original global index
hit = idx.locate(original_idx)
print(f"X_parsed[0] ← X[{original_idx}] ← token '{hit.token_str}' "
      f"(ID {hit.token_id}, freq {freq[hit.token_id]:,})")

# ── SIF weighting ──────────────────────────────────────────────────────
# On the full matrix:
X_sif = idx.sif_weight(X, idx.all_token_ids(), a=1e-3)

# On the filtered matrix — note the token_id slice via parsed_to_original:
X_parsed_sif = idx.sif_weight(
    result.X_parsed,
    idx.all_token_ids()[result.parsed_to_original],
    a=1e-3,
)
print("SIF-weighted X_parsed:", tuple(X_parsed_sif.shape))

# ── Plotly chart ───────────────────────────────────────────────────────
fig = idx.plot_token_frequencies(top_k=5_000, max_count_threshold=16_000)
fig.show()

# ── Standard point lookup ──────────────────────────────────────────────
hit = idx.locate(1234)
print(json.dumps(
    {k: v for k, v in asdict(hit).items() if k != "full_text"}, indent=2
))

collated: (5532685, 2304) torch.float32
Vocabulary covered: 91,128 distinct token IDs
10 most common: [(235265, 209353), (235269, 202543), (573, 195809), (578, 130412), (577, 125094), (108, 118108), (576, 108682), (476, 92579), (575, 67503), (235290, 55209)]
Filter: kept 3,454,115 / dropped 2,078,570 rows (ceiling = 16,000)
X_parsed[0] ← X[0] ← token '<bos>' (ID 2, freq 8,128)


In [26]:
garden_activations = idx.gather_token_activations(X,rank=4892)

In [29]:
print(Top_token_activations.activations.shape)
print(Top_token_activations.token_str)

U,S,Vh = torch.linalg.svd(Top_token_activations.activations.to('cuda'),full_matrices=False)

torch.Size([104, 2304])
▁gardens


In [43]:
worried_activations = idx.gather_token_activations(X,rank=4893)
print(worried_activations.activations.shape)
print(worried_activations.token_str)


torch.Size([104, 2304])
▁worried


In [71]:
B = worried_activations.activations.to('cuda') @ Vh.T

In [72]:
X = A @ torch.linalg.pinv(Z)

tensor([[ 0.1895, -0.1501, -0.7614,  ..., -0.7398,  1.9016, -5.6610],
        [-0.1376, -0.5113, -1.3453,  ...,  0.5326,  5.4624, -1.9180],
        [ 0.7963,  0.0806,  2.3600,  ...,  0.4874, -3.1052, -1.8288],
        ...,
        [ 0.9253,  1.8775,  2.2091,  ...,  0.8137, -2.2493,  1.4230],
        [-1.1678,  1.6336,  0.9688,  ..., -0.3339,  0.8302,  1.7843],
        [ 0.0552,  0.6472, -1.7810,  ..., -2.5045, -1.6965,  1.0947]],
       device='cuda:0')

In [77]:
S_worried = torch.norm(US, dim=1)  # Shape: (m,)
U_worried = US / S_worried.unsqueeze(1)  # Shape: (m, n)
print(S_worried)

tensor([ 97.4379, 101.5901, 113.7292, 115.6623, 122.5500, 122.3032, 102.0655,
        111.2488, 100.4278, 129.0525, 102.2969, 112.2310,  81.7398, 123.3020,
        112.2275, 114.6993, 109.6698, 102.8116, 101.3385,  77.2907, 131.5598,
        122.9172, 131.4401, 108.3948,  77.7778, 102.5721, 109.0946,  70.4063,
         59.8326,  89.5078, 101.8768,  83.9268, 120.4133,  73.6651, 107.4706,
         99.9102, 103.9975, 120.4565,  96.9037, 120.8552,  81.4834, 123.3827,
         84.8233,  91.0298, 114.4812,  87.9284,  94.0838, 100.5962,  89.9466,
         98.0043, 111.1765,  92.9821, 118.5313, 101.4564,  85.2623,  95.3441,
         84.2960, 114.2872,  99.9081,  92.9666, 116.5648, 123.2531, 112.4474,
        115.3969, 122.3797,  98.6342, 109.6425, 118.0113, 111.6387, 113.2568,
         88.2914,  98.7149,  99.1146,  94.2344,  79.6190, 135.0943, 119.6324,
        118.1037,  93.9210, 124.9820, 114.5226, 123.0779,  97.4344,  77.6462,
        100.6446,  58.2597,  96.2079,  90.1396, 110.3012, 133.42

In [53]:
worried_activations.activations.to('cuda') - U_worried * S_worried @ Vh

tensor([[ 0.0393,  0.0760, -0.7427,  ..., -0.5979,  1.9914, -5.2656],
        [-0.3596, -0.2903, -1.4730,  ...,  0.6607,  5.4489, -1.6463],
        [ 0.7470,  0.1306,  2.0402,  ...,  0.8202, -3.4077, -1.5955],
        ...,
        [ 1.0927,  2.3765,  1.9726,  ...,  1.2220, -2.0364,  1.6579],
        [-1.2998,  1.9048,  0.8476,  ..., -0.0721,  0.7961,  1.9718],
        [ 0.0263,  0.9650, -2.0221,  ..., -2.0286, -1.6670,  1.4761]],
       device='cuda:0')

In [ ]:
first = idx.gather_index_activations(X,rank=4893)
